# Audio Signal

### Waveform plot and a brief list of signal properties (sample rate, duration, sample count, and content description).

- Sample rate: 48000 kHz
- Sample count: 948000 samples
- Duration: 19.75 seconds
- Content: An excerpt of a song I'm currently working on. It does not have a title yet. More of the song than is in this excerpt exists, included in this repo as "untitled_full_so_far.wav"

![](waveform.png)

### 2. PSD plot (dB vs. Hz) and a written description of the spectral distribution.

![](psd.png)

The excerpt is relatively bass-heavy, but nothing crazy. It has a peak around 80 Hz that likely represents the fundamental for the bass - this excerpt is only over one chord, so there should not be multiple fundamentals in this plot. The peak is likely a bit over 80, as this section is in the key of F minor, and low F is at around 87.3 Hz. The higher peak at around 400 roughly corresponds to the fundamentals of the synth lead, which in this excerpt spans D3 to C4, which is about 400 $\pm$ 100 Hz. Note that I'm using note names taken from Ableton's MIDI editor, some definitions would use D4 to C5 here. 

# Designing and Analyzing a Bandpass Filter

### 3. Filter type, design method, order, and both cutoff frequencies in Hz and in radians per sample.

Based on some expirementation in Ableton, I think a bandpass filter with cutoffs around 390 Hz (0.0511 rad/sample) and 3030 Hz (0.3966 rad/sample). I'll use a chebyshev II filter for flat passband response without excessive order. While in Ableton I was working with only frequency and Q, I'm using the scipy.signal library's cheb2ord function to determine a filter order (which seems more appropriate than arbitrarily picking one). I gave it stopband frequencies of 350 Hz and 3500 Hz, maximum passband attenuation of 0.05 dB, and minimum stopband attenuation of 40 dB, which produces a 15th order filter. This seems a bit high for realtime use, but for this application it shouldn't be an issue.

### 4. Pole-zero diagram with the unit circle, and a written explanation of the pole clustering and the role of zero placement in producing the stopbands at DC and near Nyquist.

![](pole_zero.png)

Zeros produce stopbands at Nyquist because there is a zero at -1, which on a pole-zero plot corresponds to the Nyquist frequency. While difficult to make out, the zeros at 1 will be on the unit circle, and closer to 1 than the poles. Since 1 corresponds to DC, a zero at 1 will block all DC.  
  
As for the other poles and zeros, the poles are grouped so that everything within the range specified (0.0511 to 0.3966 rad) will have an equal effect from all poles and zeros, producing a flat response in this range. Outside of this range the zeros dominate, and will produce an attenuated response (albiet not a smooth one as in the passband).

### 5. Magnitude response (dB) with both −3 dB cutoff frequencies annotated.

![](mag.png)

### 6. Three-panel figure (phase response, group delay, impulse response). Within the passband from fL to fH, is the group delay approximately constant? If so, state its value in samples. Does the impulse response terminate exactly or decay toward zero, and what does that tell you about whether the filter is FIR or IIR?

![](phase.png)

### Within the passband from fL to fH, is the group delay approximately constant? If so, state its value in samples. 

While the passband does not have constant group delay (it increases the closer it gets to the transition bands), it does have a broad span of approximately 18 samples of group delay around the 1-1.5 kHz band. 

### Does the impulse response terminate exactly or decay toward zero, and what does that tell you about whether the filter is FIR or IIR?

The impulse response decays toward zero, and while it will get arbitrarily close, it is technically infinite, thus an IIR filter.

### 7. Overlaid PSD plot (input and output on the same axes) and a written perceptual description of the bandpass-filtered audio.

![](psd_overlap.png)

While it sounds subtly different from the one made in Ableton, it sounds close enough that the result is as desired! It removes most of the bass and hi-hat content and focuses on the lead. The electric piano and the distorted high-mids of the bass are also quite prominent. I feel as though I am in an elevator. Looking at the spectrum in Ableton, it seems as though the filter I created here has a narrower transition band at the cost of less stopband attenuation.

# Perceptual Thresholds of the Cutoff Frequencies

### 8. Written description of how the audio changes as $f_L$ varies, with a specific threshold value identified and a perceptual characterization of the effect on either side of it.

- $f_L$ = 50 Hz: Virtually no bass attenuation.
- $f_L$ = 200 Hz: Noticeable bass attenuation, similar but not quite as strong as in the original filter. Biggest difference is that the snares' and toms' fundamentals are now present.
- $f_L$ = 500 Hz: Similar to the original filter, but snare and electric piano/piano are more attenuated.
- $f_L$ = 1500 Hz: Very small passband, pretty much all that is audible are the lead, high-frequency bass content, and some snare content. The occasional hi-hat hiss also gets through. This range is most of the "honky" sound, think nasal voices or a saxophone, and that is definitely reflected in the lead here. 

### 9. Written description of how the audio changes as $f_H$ varies, with a specific threshold value identified and a perceptual characterization of the effect.

- $f_H$ = 18000 Hz: Virtually no treble attenuation.
- $f_H$ = 10000 Hz: Treble attenuation is noticable, but each instrument is still recognizable for what it is. Sounds a bit like a downsampled version of the song.
- $f_H$ = 5000 Hz: Similar to the original filter, but contains much more hi-hat content, and snare comes through much punchier. 
- $f_H$ = 1500 Hz: Very attenuated treble. However, some quiet high-frequency content has not been attenuated, if you listen you can hear the upper portion of the hi-hats still (maybe 12+ kHz?)

# Reverb and Echo

### 10. Difference equation or H(z) with all parameters specified, and a brief justification of the design choices.

I made a delay with adjustable parameters for number of taps, delay interval, attenuation constant, and dry/wet. It works by creating an attenuated series of impulses, where the first impulse is zero to prevent dry signal from bleeding into the delayed signal. Each successive impulse is filtered by a bandpass with increasingly tight constraints, given by: 
$$
    wp=[min(100 + 20 * (i + 1), 800), max(4000 - (i * 400), 1000)]
$$
$$
    ws=[min(50 + 15 * (i + 1), 600), max(8000 - (i * 800), 1500)]
$$
Where i is the tap number.   
  
The effect is applied by convolution with the original signal and then attenuation of the original and the delay proportional to the dry/wet parameter.

On a short delay with little attenuation the effect acts a phaser of sorts (I used TAP_SPACING = 0.01 * SAMPLE_RATE with 4 taps, an attenuation of 0.95, and 95% wet).   
With longer delay and increased attenuation it acts as an echo (TAP_SPACING = 1 * SAMPLE_RATE, 4 taps, 0.5 attenuation, 50% wet). This sounds quite bad when applied to the song I supplied, but I imagine something with less movement (especially without drums) it could sound passable. Maybe.
I was able to get a pseudo-spring reverb effect with settings of TAP_SPACING = 0.03, 15 taps, and attenuation of 0.92. Reverb effects seem to be more difficult with my current setup.

The transfer function for any of these systems is too long to fit neatly in a closed form in this box, so I created a function to return the z-transform for any impulse response and input combination:  
  
def transfer_function(h, z):  
    &nbsp;&nbsp;&nbsp;&nbsp;H = 0  
    &nbsp;&nbsp;&nbsp;&nbsp;for n in range(len(h)):  
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;H += h[n] * (z ** (-n))  
    &nbsp;&nbsp;&nbsp;&nbsp;return H 

For example, using the parameters TAP_SPACING = 1 * SAMPLE_RATE, 4 taps, 0.5 attenuation, 50% wet, some sample transfer function outputs $H(e^0j), H(e^1j),$ and $H(e^2j)$ evaluate to $2.136e-06+0j, -0.00012+0.0012j,$ and $0.00753-0.0033j$

### 11. Impulse response plot and a written explanation of the discrete-time implementation advantage.

![](delay_ir.png)

There are around 6 significant impulses visible, I say around because it depends on what is significant in the application. The big advantage of discrete time over continuous time is that real impulses are actually possible in this domain! However, in my case I have filtered my impulses and so the impulses are not pure, as shown below:

![](delay_ir_zoom.png)

### 12. Two-panel figure (magnitude in dB and phase) for the reverb filter, and a written description of the magnitude response shape.

![](delay_mag_phase.png)

This response is extremely chaotic and definitely exhibits combing, increasing in frequency as frequency increases. I also created a graph of a much simpler delay with 1 tap, no attenuation, and 1 second delay time, but it is exactly the same. If I remove the filtering and keep the other parameters simple, the resulting graph is also the same. This makes me think that there is some math error somewhere, but I do not know where it is and unfortunately cannot dedicate the time to 100% verify that my graphs are correct. So for now I will stick with the observation that there is a lot of combing.